# 數值優化技術與方法

## 學習目標

完成本 Notebook 後，你將能夠：

1. 說明機器學習訓練為何可以視為數值最佳化問題。
2. 理解目標函數、決策變數、可行域、凸性與可導性的角色。
3. 比較 MSE、MAE 與 Huber Loss 對異常值的敏感度。
4. 使用 Python 實作梯度下降法，觀察學習率對收斂行為的影響。
5. 比較 Batch Gradient Descent、SGD、Mini-batch SGD 與 Momentum 的更新差異。
6. 透過簡單的線性迴歸任務，連結損失函數、梯度與參數更新的實務流程。

## 情境設定

在機器學習中，模型訓練的核心任務是：找到一組參數，讓模型預測結果與真實資料之間的誤差最小。這正是一個數值最佳化問題。

本章將用輕量的迴歸任務示範：如何定義損失函數、如何更新模型參數，以及不同優化方法對收斂速度與穩定性的影響。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需套件，並建立一組可重複使用的簡單線性資料。這組資料用來模擬機器學習中的模型訓練情境。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# 建立簡單線性資料：y = 3x + 2 + noise
X = np.linspace(0, 10, 80)
noise = np.random.normal(0, 2, size=X.shape)
y = 3 * X + 2 + noise

print('資料筆數:', len(X))
print('X 前 5 筆:', np.round(X[:5], 2))
print('y 前 5 筆:', np.round(y[:5], 2))

plt.figure(figsize=(7, 4))
plt.scatter(X, y, alpha=0.75, label='訓練資料')
plt.xlabel('X')
plt.ylabel('y')
plt.title('簡單線性迴歸資料')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 核心概念說明

在機器學習中，訓練模型通常可以整理成以下最佳化結構：

| 組成要素 | 意義 | 在機器學習中的例子 |
|---|---|---|
| 目標函數 | 要最小化或最大化的函數 | MSE、交叉熵損失、排序損失 |
| 決策變數 | 要被調整的參數 | 權重、偏差、神經網路參數 |
| 可行域 | 參數允許存在的範圍 | 非負限制、總和為 1、正則化約束 |
| 函數性質 | 影響求解難度的數學特性 | 凸性、可導性、是否存在局部最小值 |

以線性迴歸為例，模型可寫成：

```text
預測值 = w * x + b
```

其中 `w` 與 `b` 是決策變數。訓練的目標，是找到一組 `w` 與 `b`，讓預測值與真實答案之間的損失最小。

常見損失函數包含：

- MSE：放大大誤差，對異常值敏感。
- MAE：對所有誤差給予線性懲罰，較不受異常值影響。
- Huber Loss：小誤差時像 MSE，大誤差時像 MAE，兼顧穩定與抗雜訊。

若損失函數可導，就能計算梯度，並使用梯度下降法更新參數。


In [ ]:
# ── 示範：比較 MSE、MAE 與 Huber Loss ──────────────
# 這段程式碼示範不同損失函數如何懲罰預測誤差。當誤差變大時，MSE 的懲罰成長最快，因此對異常值較敏感。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

errors = np.linspace(-10, 10, 300)

def mse_loss(e):
    return e ** 2

def mae_loss(e):
    return np.abs(e)

def huber_loss(e, delta=2.0):
    return np.where(np.abs(e) <= delta, 0.5 * e ** 2, delta * (np.abs(e) - 0.5 * delta))

loss_df = pd.DataFrame({
    'error': errors,
    'MSE': mse_loss(errors),
    'MAE': mae_loss(errors),
    'Huber': huber_loss(errors)
})

print(loss_df.iloc[[0, 75, 150, 225, 299]].round(2))

plt.figure(figsize=(8, 5))
plt.plot(errors, loss_df['MSE'], label='MSE')
plt.plot(errors, loss_df['MAE'], label='MAE')
plt.plot(errors, loss_df['Huber'], label='Huber Loss')
plt.xlabel('預測誤差 y_pred - y_true')
plt.ylabel('損失值')
plt.title('不同損失函數對誤差的懲罰方式')
plt.ylim(0, 45)
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# ── 示範：手刻梯度下降法 ──────────────────────────────
# 這段程式碼以線性迴歸為例，示範如何透過梯度下降法最小化 MSE。每一次迭代都會根據梯度調整權重 w 與偏差 b。

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.linspace(0, 10, 80)
y = 3 * X + 2 + np.random.normal(0, 2, size=X.shape)

w = 0.0
b = 0.0
learning_rate = 0.01
epochs = 120
loss_history = []

for epoch in range(epochs):
    y_pred = w * X + b
    error = y_pred - y
    loss = np.mean(error ** 2)
    loss_history.append(loss)

    grad_w = 2 * np.mean(error * X)
    grad_b = 2 * np.mean(error)

    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

print('訓練後 w:', round(w, 3))
print('訓練後 b:', round(b, 3))
print('最後 MSE:', round(loss_history[-1], 3))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('梯度下降的損失變化')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X, y, alpha=0.7, label='訓練資料')
plt.plot(X, w * X + b, color='red', label='學到的線性模型')
plt.xlabel('X')
plt.ylabel('y')
plt.title('模型擬合結果')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 常見優化演算法比較

不同優化演算法的差異，主要來自「每次用多少資料估計梯度」以及「是否調整更新方向或步長」。

| 方法 | 每次更新使用資料 | 優點 | 限制 |
|---|---:|---|---|
| Batch Gradient Descent | 全部資料 | 更新穩定，方向準確 | 資料大時計算慢 |
| SGD | 單筆資料 | 更新快，適合線上學習 | 波動大，收斂不穩 |
| Mini-batch SGD | 一小批資料 | 效率與穩定性折衷 | 批次大小需調整 |
| Momentum | 小批次或全批次 | 減少震盪，加速收斂 | 需設定動量係數 |

在實務中，深度學習常使用 Mini-batch SGD 搭配 Momentum、Adam 或 RMSprop 等進階方法。它們的核心仍然是根據梯度調整參數，只是加入了更聰明的更新策略。


In [ ]:
# ── 實際應用：比較 Batch、SGD、Mini-batch 與 Momentum ─
# 這段程式碼在同一個線性迴歸任務上比較四種更新方法。觀察損失曲線可以看出，SGD 波動較大，Mini-batch 較平衡，Momentum 則常能讓收斂更平順。

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
X = np.linspace(0, 10, 100)
y = 3 * X + 2 + np.random.normal(0, 2, size=X.shape)

X_mean = X.mean()
X_std = X.std()
X_scaled = (X - X_mean) / X_std

def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

def train(method='batch', epochs=80, lr=0.05, batch_size=16, momentum=0.9):
    rng = np.random.default_rng(7)
    w, b = 0.0, 0.0
    vw, vb = 0.0, 0.0
    losses = []

    for epoch in range(epochs):
        if method == 'batch':
            batches = [np.arange(len(X_scaled))]
        elif method == 'sgd':
            batches = [[i] for i in rng.permutation(len(X_scaled))]
        else:
            idx = rng.permutation(len(X_scaled))
            batches = [idx[i:i + batch_size] for i in range(0, len(idx), batch_size)]

        for batch in batches:
            xb = X_scaled[batch]
            yb = y[batch]
            pred = w * xb + b
            error = pred - yb
            grad_w = 2 * np.mean(error * xb)
            grad_b = 2 * np.mean(error)

            if method == 'momentum':
                vw = momentum * vw + lr * grad_w
                vb = momentum * vb + lr * grad_b
                w -= vw
                b -= vb
            else:
                w -= lr * grad_w
                b -= lr * grad_b

        full_pred = w * X_scaled + b
        losses.append(mse(y, full_pred))

    return w, b, losses

methods = {
    'Batch GD': {'method': 'batch', 'lr': 0.05},
    'SGD': {'method': 'sgd', 'lr': 0.01},
    'Mini-batch SGD': {'method': 'mini_batch', 'lr': 0.03, 'batch_size': 16},
    'Momentum': {'method': 'momentum', 'lr': 0.03, 'batch_size': 16, 'momentum': 0.9}
}

results = {}
for name, params in methods.items():
    w, b, losses = train(**params)
    results[name] = {'w': w, 'b': b, 'losses': losses}
    print(f'{name:14s} 最後 MSE: {losses[-1]:.3f}')

plt.figure(figsize=(9, 5))
for name, result in results.items():
    plt.plot(result['losses'], label=name)

plt.xlabel('Epoch')
plt.ylabel('完整資料集 MSE')
plt.title('四種梯度更新方法的損失曲線比較')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
